In [ ]:
# ===== 設定（ここだけ変更すればOK） =====
CSV_PATH = "monthly_1950-01_2020-07.csv"  # 月次データ
TEST_RATIO = 0.2
SEED = 0
# ========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ----------------------------
# 1. データ読み込み（月次）
# ----------------------------
df = pd.read_csv(CSV_PATH)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").set_index("Date")

y = df["Price"].astype(float)

# ----------------------------
# 2. ARなしの特徴量（seasonality + trend）
# ----------------------------
d = pd.DataFrame(index=df.index)
d["y"] = y

# 年・月・トレンド
d["year"] = d.index.year
d["month"] = d.index.month
d["t"] = np.arange(len(d))

# 月の周期性 sin/cos
month_rad = 2 * np.pi * (d["month"] - 1) / 12.0
d["month_sin"] = np.sin(month_rad)
d["month_cos"] = np.cos(month_rad)

d = d.dropna()

X = d.drop(columns=["y"]).values
target = d["y"].values
feature_names = list(d.drop(columns=["y"]).columns)

# ----------------------------
# 3. 時系列分割（過去→未来）
# ----------------------------
n = len(d)
split = int(n * (1 - TEST_RATIO))

X_train, X_test = X[:split], X[split:]
y_train, y_test = target[:split], target[split:]

dates = d.index
dates_train, dates_test = dates[:split], dates[split:]

# ----------------------------
# 4. RandomForest 回帰
# ----------------------------
model = RandomForestRegressor(
    n_estimators=500,
    random_state=SEED,
    n_jobs=-1,
)
model.fit(X_train, y_train)

pred_train = model.predict(X_train)
pred_test = model.predict(X_test)

# ----------------------------
# 5. 評価表示
# ----------------------------
def print_metrics(name, yt, yp):
    mae = mean_absolute_error(yt, yp)
    rmse = np.sqrt(mean_squared_error(yt, yp))
    r2 = r2_score(yt, yp)
    print(f"=== {name} ===")
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R2  :", r2)
    print()

print_metrics("Train", y_train, pred_train)
print_metrics("Test ", y_test, pred_test)

# ----------------------------
# 6. 特徴量重要度
# ----------------------------
importances = model.feature_importances_
imp_df = pd.DataFrame({"feature": feature_names, "importance": importances})
imp_df = imp_df.sort_values("importance")

# ----------------------------
# 7. プロット1：実測 vs 予測（時系列）
# ----------------------------
plt.figure(figsize=(12, 5))
plt.plot(dates_train, y_train, label="Train True", linewidth=1)
plt.plot(dates_test, y_test, label="Test True", linewidth=2)
plt.plot(dates_test, pred_test, "--", label="Test Pred (RF)", linewidth=2)
plt.title("True vs Predicted (RandomForest, Monthly Data)")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.tight_layout()
plt.show()

# ----------------------------
# 8. プロット2：特徴量重要度
# ----------------------------
plt.figure(figsize=(8, 5))
plt.barh(imp_df["feature"], imp_df["importance"])
plt.title("RandomForest Feature Importances (Monthly Regression)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()